# Generic Research Run Viewer

Reusable viewer for any run directory that contains:
- `summary.csv`
- optional `by_date.csv`
- optional `params.json` and `manifest.json`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 200)
plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
RUNS_ROOT = Path('../research/output')
SUMMARY_NAME = 'summary.csv'
BY_DATE_NAME = 'by_date.csv'

run_dirs = []
if RUNS_ROOT.exists():
    run_dirs = sorted(
        [d for d in RUNS_ROOT.iterdir() if d.is_dir() and (d / SUMMARY_NAME).exists()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

recent = run_dirs[:3]
display(Markdown('## Top 3 Recent Runs'))
display(pd.DataFrame([
    {
        'run_tag': d.name,
        'modified_time': pd.to_datetime(d.stat().st_mtime, unit='s').strftime('%Y-%m-%d %H:%M:%S'),
        'has_by_date': (d / BY_DATE_NAME).exists(),
        'path': str(d),
    }
    for d in recent
]) if recent else pd.DataFrame({'message': ['No run directories found']}))

RUN_TAG = None  # Optional override
SELECTED_RUN = RUN_TAG or (recent[0].name if recent else None)
if SELECTED_RUN is None:
    raise FileNotFoundError('No run found. Generate research outputs first.')

RUN_DIR = RUNS_ROOT / SELECTED_RUN
print(f'Using run: {SELECTED_RUN}')
print(f'Run dir: {RUN_DIR.resolve()}')


In [ ]:
summary = pd.read_csv(RUN_DIR / SUMMARY_NAME)
by_date_path = RUN_DIR / BY_DATE_NAME
by_date = pd.read_csv(by_date_path, parse_dates=['date']) if by_date_path.exists() else None

params_path = RUN_DIR / 'params.json'
manifest_path = RUN_DIR / 'manifest.json'
params = json.loads(params_path.read_text(encoding='utf-8')) if params_path.exists() else {}
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}

display(Markdown('## Summary'))
display(summary)
if params:
    display(Markdown('## Params'))
    display(pd.DataFrame({'parameter': list(params.keys()), 'value': list(params.values())}))
if manifest:
    display(Markdown('## Manifest'))
    display(pd.DataFrame({'field': list(manifest.keys()), 'value': [manifest[k] for k in manifest.keys()]}))


## Plot Helpers

The notebook auto-plots common columns if they exist.


In [ ]:
def first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None

h_col = first_existing(summary, ['horizon_days', 'horizon', 'h'])
x = summary[h_col].astype(str) if h_col else summary.index.astype(str)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

left_cols = [c for c in ['implied_ir', 'realized_active_ir', 'sharpe', 'total_return'] if c in summary.columns]
for c in left_cols[:2]:
    axes[0].plot(x, summary[c], marker='o', label=c)
axes[0].set_title('Performance / IR Metrics')
axes[0].legend()

mid_cols = [c for c in ['avg_rank_ic', 'ic_ir', 'ic_std'] if c in summary.columns]
for c in mid_cols:
    axes[1].plot(x, summary[c], marker='o', label=c)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('IC Diagnostics')
axes[1].legend()

right_cols = [c for c in ['avg_tc_proxy', 'avg_breadth_proxy', 'signal_coverage'] if c in summary.columns]
for c in right_cols:
    axes[2].plot(x, summary[c], marker='o', label=c)
axes[2].set_title('Implementation / Breadth / Coverage')
axes[2].legend()

fig.suptitle(f'Run: {SELECTED_RUN}', y=1.05)
fig.tight_layout()
plt.show()


In [ ]:
if by_date is None or by_date.empty:
    display(Markdown('No `by_date.csv` available for this run.'))
else:
    d = by_date.copy().sort_values('date')
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

    for h, g in d.groupby('horizon_days' if 'horizon_days' in d.columns else d.columns[0]):
        g = g.sort_values('date')
        if 'active_return' in g.columns:
            axes[0].plot(g['date'], g['active_return'].cumsum(), label=f'h={h}')
        if 'ic' in g.columns:
            axes[1].plot(g['date'], g['ic'].rolling(20, min_periods=5).mean(), label=f'h={h}')
        if 'implied_ir_component' in g.columns:
            axes[2].plot(g['date'], g['implied_ir_component'], label=f'h={h}', alpha=0.9)

    axes[0].set_title('Cumulative Active Return')
    axes[1].set_title('Rolling IC Mean (20)')
    axes[2].set_title('Implied IR Component')
    axes[2].axhline(0, color='black', linewidth=1)

    for ax in axes:
        ax.legend(loc='best')
    fig.tight_layout()
    plt.show()
